# Atelier 3.B — L'agent du service après-vente ★

**Durée : 75 min** · Projet fil rouge, incrément du Jour 3

Le tri (Jour 1) et l'assistant documentaire (Jour 2) savent **répondre** ; l'agent sait **agir**. Il traite une demande de remboursement de bout en bout :

1. il consulte le CRM de Fictiva, exposé par un **serveur MCP** ;
2. il calcule le montant avec un outil déterministe qui applique la politique de retours ;
3. il crée un ticket, après **validation humaine** ;
4. il rédige un brouillon de réponse au client.

L'agent est un graphe **LangGraph** ; les briques sont dans `agents_fictiva.py`, `crm.py` et `serveur_crm_mcp.py`.

In [1]:
import json
import sys

import pandas as pd
from langchain_core.messages import HumanMessage
from langgraph.types import Command

import agents_fictiva as af
import crm
import outils_llm as llm

MODELE = llm.choisir_modele()   # le modèle doit savoir appeler des outils
print("Modèle :", MODELE, "| date du scénario :", crm.DATE_REFERENCE)

def formater(demande):
    return f"De : {demande['expediteur']} ({demande['entreprise']})\nObjet : {demande['objet']}\n\n{demande['corps']}"

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Modèle : groq:openai/gpt-oss-20b | date du scénario : 2026-10-23


## 1. Le CRM simulé

Clients, commandes et catalogue de Fictiva, cohérents avec les emails du Jour 1 et les documents du Jour 2. La date du scénario est fixée au 23 octobre 2026 pour que les délais de retour restent reproductibles.

In [2]:
pd.DataFrame([{"numero_client": n, **c} for n, c in crm.DONNEES["clients"].items()])

,numero_client,nom,contact,statut,ville
0,CL-10012,Roux Avocats,Olivier Roux,premium,Lausanne
1,CL-10234,Rochat Emballages SA,Marc Rochat,standard,Crissier
2,CL-10345,Keller & Fils SA,Daniel Keller,standard,Fribourg
3,CL-10456,Chevalier Conseil,Émilie Chevalier,standard,Genève
4,CL-10520,Simon Graphisme,Valérie Simon,standard,Neuchâtel
5,CL-10590,Atelier Morel Sàrl,Nadia Morel,standard,Lausanne
6,CL-10610,Blanc Design,Laura Blanc,standard,Vevey
7,CL-10730,Jacot Formation,Sarah Jacot,standard,Yverdon-les-Bains
8,CL-10812,Golay Architectes,Martine Golay,standard,Morges
9,CL-10901,Bureau Technique Aubert SA,Alain Aubert,premium,Lausanne


In [3]:
pd.DataFrame([{"commande": n, "client": c["client"], "statut": c["statut"], "livraison": c["date_livraison"],
               "articles": ", ".join(f"{l['quantite']} × {l['reference']}" for l in c["lignes"])}
              for n, c in crm.DONNEES["commandes"].items()])

,commande,client,statut,livraison,articles
0,CMD-2023-0150,CL-10901,livree,2023-06-12,1 × NS-4
1,CMD-2026-0331,CL-10012,livree,2026-09-08,1 × LX-500
2,CMD-2026-0368,CL-10610,livree,2026-09-18,1 × EC-27Q
3,CMD-2026-0390,CL-10812,en_retard,NaN,12 × NB-14
4,CMD-2026-0395,CL-10590,livree,2026-10-01,"5 × EC-27Q, 5 × DK-200"
5,CMD-2026-0402,CL-10345,livree,2026-10-14,10 × SE-700
6,CMD-2026-0409,CL-10520,livree,2026-10-12,20 × CE-20
7,CMD-2026-0412,CL-10234,livree,2026-10-19,3 × LX-500
8,CMD-2026-0415,CL-10456,livree,2026-10-09,3 × KB-80
9,CMD-2026-0418,CL-10730,livree,2026-10-16,2 × WC-HD


## 2. Le serveur MCP

Le **Model Context Protocol** standardise la façon dont un agent découvre et appelle des outils. Le serveur du CRM n'expose que trois outils **en lecture** : c'est le principe du moindre privilège.

In [4]:
print(open("serveur_crm_mcp.py", encoding="utf-8").read())

"""
serveur_crm_mcp.py : serveur MCP qui expose le CRM de Fictiva en lecture seule.

Le serveur communique par l'entrée et la sortie standard (transport stdio) :
c'est le client MCP (l'agent, un IDE, un assistant de bureau) qui le lance.
Test manuel : `mcp dev serveur_crm_mcp.py` ouvre l'inspecteur MCP.
"""

from mcp.server.fastmcp import FastMCP

import crm

serveur = FastMCP("crm-fictiva")


@serveur.tool()
def rechercher_client(numero_client: str = "", nom: str = "") -> dict:
    """Retrouve la fiche d'un client de Fictiva : nom de l'entreprise, contact, statut
    (standard ou premium), ville. Chercher par numéro de client (format CL-xxxxx) ou,
    à défaut, par une partie du nom de l'entreprise."""
    return crm.rechercher_client(numero_client, nom)


@serveur.tool()
def commandes_du_client(numero_client: str) -> dict:
    """Liste les commandes d'un client (numéro, date, statut, références commandées).
    Utile quand le client ne cite pas son numéro de commande."""
    return c

N'importe quel client MCP peut utiliser ce serveur : un agent, un IDE, un assistant de bureau. Voici le dialogue du protocole, sans framework : on lance le serveur, on liste ses outils, on en appelle un.

In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

parametres = StdioServerParameters(command=sys.executable, args=["serveur_crm_mcp.py"])
async with stdio_client(parametres) as (lecture, ecriture):
    async with ClientSession(lecture, ecriture) as session:
        await session.initialize()
        for outil in (await session.list_tools()).tools:
            print(f"{outil.name} {list(outil.inputSchema['properties'])}")
        resultat = await session.call_tool("consulter_commande", {"numero_commande": "CMD-2026-0415"})
        print("\n", resultat.content[0].text[:500])

rechercher_client ['numero_client', 'nom']
commandes_du_client ['numero_client']
consulter_commande ['numero_commande']

 {
  "numero_commande": "CMD-2026-0415",
  "client": {
    "numero_client": "CL-10456",
    "nom": "Chevalier Conseil",
    "statut": "standard"
  },
  "statut": "livree",
  "date_commande": "2026-10-07",
  "date_livraison": "2026-10-09",
  "lignes": [
    {
      "reference": "KB-80",
      "quantite": 3,
      "prix_unitaire": 80.0,
      "designation": "Clavier sans fil",
      "categorie": "materiel"
    }
  ],
  "remise_pct": 0,
  "frais_livraison": 25.0,
  "total_ht": 240.0,
  "facture": "F


## 3. Les outils locaux

- `calculer_remboursement` applique la politique de retours **en Python**. Le calcul d'un montant ne se confie pas au modèle : un outil déterministe est exact, testable et auditable.
- `creer_ticket` écrit dans le CRM. C'est une action engageante : l'outil **suspend** l'agent (`interrupt`) jusqu'à la décision d'un opérateur.

In [6]:
print(crm.calculer_remboursement("CMD-2026-0415", ["KB-80"], defectueux=False))
print(crm.calculer_remboursement("CMD-2026-0402", ["SE-700"], defectueux=False))
print(crm.calculer_remboursement("CMD-2026-0409", ["CE-20"], defectueux=False, consommables_ouverts=True))

{'remboursable': True, 'montant_chf': 216.0, 'validation_requise': False, 'motif': 'Retour sans défaut dans le délai de 30 jours : frais de traitement de 24.00 CHF déduits.', 'articles_politique': ['1.1', '3']}
{'remboursable': True, 'montant_chf': 6250.0, 'validation_requise': True, 'motif': 'Retour sans défaut dans le délai de 30 jours : frais de traitement de 50.00 CHF déduits.', 'articles_politique': ['1.1', '3', '5']}
{'remboursable': False, 'montant_chf': 0.0, 'validation_requise': False, 'motif': 'Articles non repris sans défaut (personnalisés ou consommables ouverts) : CE-20.', 'articles_politique': ['3.4', '4']}


## 4. Le graphe de l'agent

Deux nœuds : `agent` (le modèle, avec la liste des outils) et `outils` (l'exécution). Tant que le modèle demande des outils, on boucle ; dès qu'il répond sans outil, on termine. La mémoire (*checkpointer*) conserve l'état de chaque conversation : c'est elle qui permet la pause et la reprise.

In [7]:
outils_mcp = await af.outils_crm_mcp()
OUTILS = outils_mcp + [af.calculer_remboursement, af.creer_ticket]
print([o.name for o in OUTILS])

agent = af.construire_agent(OUTILS, modele=MODELE)
print(agent.get_graph().draw_mermaid())

['rechercher_client', 'commandes_du_client', 'consulter_commande', 'calculer_remboursement', 'creer_ticket']
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	outils(outils)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -. &nbsp;tools&nbsp; .-> outils;
	outils --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Le prompt système donne la **méthode**, pas les réponses :

In [8]:
print(af.PROMPT_SAV)

Tu es l'agent du service après-vente de Fictiva SA, distributeur B2B de matériel informatique.
Tu traites une demande de client reçue par email. Nous sommes le 23 octobre 2026.

Méthode :
1. Identifie le client et la commande avec les outils du CRM. Ne suppose jamais une information que tu peux vérifier. Si l'email ne donne pas le numéro de commande, cherche le client puis ses commandes.
2. Pour un retour ou un remboursement, calcule le montant uniquement avec l'outil calculer_remboursement, jamais de tête. Déduis de l'email si l'article est défectueux et, pour des consommables, s'ils ont été ouverts.
3. Crée un ticket qui résume la demande, la décision et le montant éventuel.
4. Termine par un brouillon de réponse au client : 5 à 8 lignes, vouvoiement, sans rien promettre au-delà de la politique. Si une validation par un responsable est requise, dis que la demande est en cours de validation.

Si une information reste introuvable, dis-le dans le brouillon au lieu de l'inventer.


## 5. Traiter une demande, avec validation humaine

In [9]:
demandes = af.charger_demandes()
demande = demandes[0]
print(formater(demande))

configuration = af.config(demande["id"])
resultat = await agent.ainvoke({"messages": [HumanMessage(formater(demande))]}, configuration)

De : Émilie Chevalier (Chevalier Conseil)
Objet : Demande de remboursement – claviers non utilisés

Bonjour,

Nous avons commandé par erreur 3 claviers KB-80 (commande CMD-2026-0415, livrée le 9 octobre). Ils sont encore dans leur emballage d'origine. Nous souhaitons les retourner et être remboursés. Merci de nous indiquer la marche à suivre.

Client CL-10456
Émilie Chevalier, Chevalier Conseil


L'agent s'est arrêté avant de créer le ticket. L'opérateur voit exactement ce qui sera écrit :

In [10]:
for interruption in resultat.get("__interrupt__", []):
    print(json.dumps(interruption.value, ensure_ascii=False, indent=2))

{
  "action": "creer_ticket",
  "numero_client": "CL-10456",
  "categorie": "remboursement",
  "resume": "Client demande le remboursement de 3 claviers KB-80 (commande CMD-2026-0415) non utilisés et dans l’emballage d’origine. Politique de retour acceptée, remboursement de 216.0 CHF (240.0 CHF HT – 24.0 CHF frais de traitement).",
  "montant_chf": 216.0
}


In [11]:
resultat = await agent.ainvoke(Command(resume="oui"), configuration)   # l'opérateur valide
af.afficher_trajectoire(resultat["messages"])
print("Tickets :", crm.TICKETS)

👤 DEMANDE
De : Émilie Chevalier (Chevalier Conseil)
Objet : Demande de remboursement – claviers non utilisés

Bonjour,

Nous avons commandé par erreur 3 claviers KB-80 (commande CMD-2026-0415, livrée le 9 octobre). Ils sont encore dans leur emballage d'origine. Nous souhaitons les retourner et être remboursés. Merci de nous indiquer la marche à suivre.

Client CL-10456
Émilie Chevalier, Chevalier Conseil

🤖 APPEL  consulter_commande({"numero_commande": "CMD-2026-0415"})
🔧 RÉSULTAT consulter_commande : {
  "numero_commande": "CMD-2026-0415",
  "client": {
    "numero_client": "CL-10456",
    "nom": "Chevalier Conseil",
    "statut": "standard"
  },
  "statut": "livree",
  "date_commande": "2026-10-07",
  "date_livraison": "2026-10-09",
  "lignes": [
    {
      "reference": "KB-80",
      "quantit

🤖 APPEL  calculer_remboursement({"consommables_ouverts": false, "defectueux": false, "numero_commande": "CMD-2026-0415", "references": ["KB-80"]})
🔧 RÉSULTAT calculer_remboursement : {"rembou

## 6. Quand l'opérateur refuse

La commande de Golay Architectes n'a jamais été livrée : ce n'est pas un retour. Voyons ce que l'agent propose, puis refusons le ticket avec une consigne.

In [12]:
demande = demandes[5]
configuration = af.config(demande["id"])
resultat = await agent.ainvoke({"messages": [HumanMessage(formater(demande))]}, configuration)
for interruption in resultat.get("__interrupt__", []):
    print(json.dumps(interruption.value, ensure_ascii=False, indent=2))

{
  "action": "creer_ticket",
  "numero_client": "CL-10812",
  "categorie": "reclamation",
  "resume": "Réclamation client pour non-livraison de la commande CMD-2026-0390. Demande de remboursement de l’acompte de 7 200 CHF. Ticket créé en attente d’achèvement de la procédure d’annulation et de validation d’un montant supérieur à 5 000 CHF.",
  "montant_chf": 7200.0
}


In [13]:
consigne = ("non : la politique de retours ne s'applique pas à une commande non livrée et aucun remboursement "\
            "ne doit être promis. Crée un ticket « reclamation » sans montant, à transmettre au responsable du service client.")
resultat = await agent.ainvoke(Command(resume=consigne), configuration)
while resultat.get("__interrupt__"):
    print("Nouvelle proposition :", json.dumps(resultat["__interrupt__"][0].value, ensure_ascii=False))
    resultat = await agent.ainvoke(Command(resume="oui"), configuration)
print("\n" + resultat["messages"][-1].content)


**Ticket créé**  
- **Client** : CL‑10812 – Golay Architectes  
- **Catégorie** : réclamation  
- **Référence de la commande** : CMD‑2026‑0390  
- **Résumé** : Demande de remboursement de l’acompte de 7 200 CHF suite à une non‑livraison.  
- **Décision** : La politique de Fictiva ne prévoit pas de remboursement de l’acompte pour une commande non livrée (statut « en_retard »). La demande sera traitée comme une réclamation et un suivi interne sera lancé pour localiser la livraison. Aucun remboursement ne sera effectué à ce stade.  
- **Montant** : CHF 0.00  

---

**Brouillon de réponse au client**

Madame Golay,

Nous avons bien reçu votre demande de remboursement concernant la commande CMD‑2026‑0390. Après vérification, la commande n’a pas encore été livrée et, conformément à notre politique, un acompte versé sur une commande non livrée ne fait pas l’objet de remboursement.  

Nous allons immédiatement lancer une enquête pour localiser votre livraison et vous tiendrons informée de l’a

**À retenir.** La validation humaine n'est pas une formalité : sur les décisions engageantes, elle rattrape les erreurs de l'agent. Ici, l'opérateur a empêché une promesse de remboursement que la politique ne prévoit pas.

Relisez pourtant le brouillon : l'agent peut encore affirmer des règles qui n'existent nulle part. Il connaît le CRM, mais **pas la documentation** de Fictiva.

## 7. Toutes les demandes

On traite les autres demandes en validant automatiquement les tickets, puis on compare les montants calculés aux montants attendus.

In [14]:
async def traiter(demande, decision="oui"):
    configuration = af.config(demande["id"] + "-lot")
    resultat = await agent.ainvoke({"messages": [HumanMessage(formater(demande))]}, configuration)
    while resultat.get("__interrupt__"):
        resultat = await agent.ainvoke(Command(resume=decision), configuration)
    return resultat

lignes = []
for demande in demandes[1:5]:
    resultat = await traiter(demande)
    calculs = [json.loads(m.content) for m in resultat["messages"] if getattr(m, "name", "") == "calculer_remboursement"]
    dernier = calculs[-1] if calculs else {}
    lignes.append({"demande": demande["id"], "attendu": demande["attendu"]["montant_chf"],
                   "calcule": dernier.get("montant_chf"), "validation_requise": dernier.get("validation_requise"),
                   "appels_d_outils": sum(len(getattr(m, "tool_calls", [])) for m in resultat["messages"])})
pd.DataFrame(lignes)

,demande,attendu,calcule,validation_requise,appels_d_outils
0,SAV-2,360.0,360.0,False,4
1,SAV-3,6250.0,6250.0,True,3
2,SAV-4,0.0,0.0,False,5
3,SAV-5,600.0,600.0,False,4


## 8. La mémoire de conversation

Le fil de conversation de la première demande est conservé : on peut poursuivre l'échange sans tout réexpliquer.

In [15]:
configuration = af.config(demandes[0]["id"])
resultat = await agent.ainvoke({"messages": [HumanMessage(
    "La cliente demande si elle peut recevoir un avoir plutôt qu'un virement. Réponds-lui en deux phrases, sans créer de nouveau ticket.")]},
    configuration)
while resultat.get("__interrupt__"):   # si l'agent tente malgré tout de créer un ticket, l'opérateur refuse
    resultat = await agent.ainvoke(Command(resume="non : aucun nouveau ticket n'est nécessaire."), configuration)
print(resultat["messages"][-1].content)

Madame Chevalier, il est possible de vous attribuer un avoir au lieu d’un virement ; il vous suffira de nous faire part de cette préférence avant que le remboursement soit traité. L’avoir sera alors crédité sur votre compte client dans les 5 jours ouvrés suivant la confirmation de retour.


**À vérifier.** L'article 6 de la politique de retours prévoit un remboursement « par virement ou sous forme d'avoir ». La réponse de l'agent est-elle conforme ? Sans accès à la documentation, l'agent répond de mémoire : s'il a juste, c'est par chance. Au Jour 4, l'assistant documentaire du Jour 2 devient l'un de ses outils.

✏️ Écrivez une demande ambiguë (un client qui ne donne ni numéro de client ni numéro de commande, et cite un produit commandé plusieurs fois). L'agent pose-t-il les bons appels ? Que manque-t-il au prompt ?

## À retenir

- Le modèle **orchestre**, les outils **exécutent** : les montants viennent d'un calcul déterministe, pas du modèle.
- MCP sépare l'agent de ses sources : le même serveur CRM peut servir d'autres agents et d'autres applications.
- Moindre privilège : lecture seule via MCP, écriture réservée à un outil soumis à **validation humaine**.
- La mémoire par fil de conversation permet la pause, la reprise et le suivi d'un dossier.